# Purpose

This notebook creates the final feature-engineered dataset for modeling future market volatility. The goal is to turn each asset's daily price and return history into features that help a model recognize periods when volatility may rise or fall.

## Feature Columns Added

- `return_lag_1`: Previous trading day's return for the same ticker. Recent returns can signal short-term momentum or reversal before volatility changes.
- `return_lag_5`: Return from five trading days earlier for the same ticker. This gives the model a slightly wider short-term memory.
- `rolling_return_5d`: Average return over the previous 5 trading days. This captures short-term trend direction.
- `rolling_return_20d`: Average return over the previous 20 trading days. This captures roughly one month of return behavior.
- `abs_return`: Absolute value of the daily return. Large moves matter for volatility whether the return is positive or negative.
- `squared_return`: Squared daily return. Squaring emphasizes unusually large moves, which are especially important for volatility modeling.
- `rolling_abs_return_20d`: Average absolute return over the previous 20 trading days. This measures recent realized movement size.
- `rolling_squared_return_20d`: Average squared return over the previous 20 trading days. This gives another realized-volatility-style signal that heavily weights large shocks.
- `rolling_volatility_5d`: Standard deviation of returns over the previous 5 trading days. This measures very recent realized volatility.
- `rolling_volatility_20d`: Standard deviation of returns over the previous 20 trading days. This measures roughly one month of realized volatility.
- `moving_avg_20d`: Average adjusted close price over the previous 20 trading days. This supports trend and relative-price features.
- `price_to_moving_avg_20d`: Current price divided by the 20-day moving average. This shows whether the asset is trading above or below its recent trend.
- `future_volatility_20d`: Target variable: the next 20 trading days of realized volatility for the same ticker.

### Samii Thoughts
I googled and I found that these features are useful because volatility is more about the size and clustering of price moves than about direction alone. Absolute returns, squared returns, and rolling standard deviations help the model focus on how unstable the asset has recently been, while lagged and rolling returns add context about recent trend behavior.

# Imports

In [42]:
import pandas as pd 
import numpy as np
from pathlib import Path

# ENUMS

In [43]:
PROCESSED_DATA_PATH = Path("../data/processed")
INTEGRATED_DATA_PATH = PROCESSED_DATA_PATH / "integrated"
FEATURE_DATA_PATH = PROCESSED_DATA_PATH / "features"

## Create Feature Data Directory

In [44]:
if not FEATURE_DATA_PATH.exists():
    FEATURE_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory {FEATURE_DATA_PATH} already exists.")

Directory ..\data\processed\features already exists.


# Modeling Base CSV Setup

In [45]:
modeling_base = pd.read_csv(
    INTEGRATED_DATA_PATH / "modeling_base_dataset.csv",
    parse_dates=["Date"]
)

## Modeling Base Shape Validation

In [46]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock


In [47]:
modeling_base.shape

(42189, 11)

In [48]:
modeling_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 42189 entries, 0 to 42188
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    42189 non-null  datetime64[us]
 1   ticker                  42189 non-null  str           
 2   adjusted_close          42189 non-null  float64       
 3   daily_return            42189 non-null  float64       
 4   risk_free_rate_pct      42189 non-null  float64       
 5   risk_free_rate_decimal  42189 non-null  float64       
 6   cpi_index               42189 non-null  float64       
 7   cpi_pct_change          42189 non-null  float64       
 8   company_name            42189 non-null  str           
 9   gics_sector             42189 non-null  str           
 10  asset_type              42189 non-null  str           
dtypes: datetime64[us](1), float64(6), str(4)
memory usage: 5.2 MB


In [49]:
modeling_base.isna().sum()

Date                      0
ticker                    0
adjusted_close            0
daily_return              0
risk_free_rate_pct        0
risk_free_rate_decimal    0
cpi_index                 0
cpi_pct_change            0
company_name              0
gics_sector               0
asset_type                0
dtype: int64

## Modeling Base Sorting 

Sorting Before Making Rolling Features

In [50]:
modeling_base = modeling_base.sort_values(['ticker','Date']).reset_index(drop=True)

In [51]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock


## Modeling Base Lagged Returns Values

In [52]:
modeling_base["return_lag_1"] = modeling_base.groupby("ticker")["daily_return"].shift(1)

In [53]:
modeling_base["return_lag_5"] = modeling_base.groupby("ticker")["daily_return"].shift(5)

In [54]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN


# Create Rolling Return Features

In [55]:
modeling_base['rolling_return_5d'] = (
    modeling_base.groupby('ticker')['daily_return']
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [56]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405


In [57]:
modeling_base['rolling_return_20d'] = (
    modeling_base.groupby('ticker')['daily_return']
    .rolling(window=20)
    .mean()
    .reset_index(level=0, drop=True)
)

In [58]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN


# Create Return Magnitude Features

Volatility depends on how large price moves are, not just whether returns are positive or negative. These features convert daily returns into movement-size signals.

In [59]:
modeling_base["abs_return"] = modeling_base["daily_return"].abs()

In [60]:
modeling_base["squared_return"] = modeling_base["daily_return"] ** 2

In [61]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,abs_return,squared_return
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,0.000174,3.035172e-08
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,0.004645,2.157600e-05
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,0.011385,1.296189e-04
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,0.003714,1.379399e-05
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.000114,1.310382e-08


# Create Rolling Return Magnitude Features

These 20-day rolling features summarize recent movement size over roughly one trading month.

In [62]:
modeling_base["rolling_abs_return_20d"] = (
    modeling_base.groupby("ticker")["abs_return"]
    .rolling(window=20)
    .mean()
    .reset_index(level=0, drop=True)
)

In [63]:
modeling_base["rolling_squared_return_20d"] = (
    modeling_base.groupby("ticker")["squared_return"]
    .rolling(window=20)
    .mean()
    .reset_index(level=0, drop=True)
)

In [64]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,0.000174,3.035172e-08,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,0.004645,2.157600e-05,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,0.011385,1.296189e-04,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,0.003714,1.379399e-05,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.000114,1.310382e-08,NaN,NaN


# Create Rolling Volatility Feature

In [65]:
modeling_base['rolling_volatility_5d'] = (
    modeling_base.groupby('ticker')['daily_return']
    .rolling(window=5)
    .std()
    .reset_index(level=0, drop=True)
)

In [66]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,NaN,NaN,NaN,NaN,0.000174,3.035172e-08,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.000174,NaN,NaN,NaN,0.004645,2.157600e-05,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock,0.004645,NaN,NaN,NaN,0.011385,1.296189e-04,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock,0.011385,NaN,NaN,NaN,0.003714,1.379399e-05,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock,-0.003714,NaN,0.002405,NaN,0.000114,1.310382e-08,NaN,NaN,0.005833


In [67]:
modeling_base["rolling_volatility_20d"] = (
    modeling_base.groupby("ticker")["daily_return"]
    .rolling(window=20)
    .std()
    .reset_index(level=0, drop=True)
)

In [68]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,...,return_lag_1,return_lag_5,rolling_return_5d,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,NaN,NaN,0.000174,3.035172e-08,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,-0.000174,NaN,NaN,NaN,0.004645,2.157600e-05,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,...,0.004645,NaN,NaN,NaN,0.011385,1.296189e-04,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,...,0.011385,NaN,NaN,NaN,0.003714,1.379399e-05,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,...,-0.003714,NaN,0.002405,NaN,0.000114,1.310382e-08,NaN,NaN,0.005833,NaN


# Create Moving Average Feature

In [69]:
modeling_base['moving_avg_20d'] = (
    modeling_base.groupby('ticker')['adjusted_close']
    .rolling(window=20)
    .mean()
    .reset_index(level=0, drop=True)
)

In [70]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,...,return_lag_5,rolling_return_5d,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,NaN,0.000174,3.035172e-08,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,NaN,0.004645,2.157600e-05,NaN,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,NaN,0.011385,1.296189e-04,NaN,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,NaN,0.003714,1.379399e-05,NaN,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,...,NaN,0.002405,NaN,0.000114,1.310382e-08,NaN,NaN,0.005833,NaN,NaN


In [71]:
modeling_base["price_to_moving_avg_20d"] = (
    modeling_base["adjusted_close"] / modeling_base["moving_avg_20d"]
)

In [72]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,...,rolling_return_5d,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,0.000174,3.035172e-08,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,0.004645,2.157600e-05,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,0.011385,1.296189e-04,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,...,NaN,NaN,0.003714,1.379399e-05,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,...,0.002405,NaN,0.000114,1.310382e-08,NaN,NaN,0.005833,NaN,NaN,NaN


# Create Future Volatility Target

In [73]:
modeling_base["future_volatility_20d"] = (
    modeling_base.groupby("ticker")["daily_return"]
    .transform(lambda returns: returns.rolling(window=20).std().shift(-20))
)

In [74]:
modeling_base.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,...,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
0,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,0.000174,3.035172e-08,NaN,NaN,NaN,NaN,NaN,NaN,0.009490
1,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,...,NaN,0.004645,2.157600e-05,NaN,NaN,NaN,NaN,NaN,NaN,0.013250
2,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,...,NaN,0.011385,1.296189e-04,NaN,NaN,NaN,NaN,NaN,NaN,0.013567
3,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,...,NaN,0.003714,1.379399e-05,NaN,NaN,NaN,NaN,NaN,NaN,0.017207
4,2018-01-09,AAPL,40.750969,-0.000114,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,...,NaN,0.000114,1.310382e-08,NaN,NaN,0.005833,NaN,NaN,NaN,0.017658


# Drop Rows That Cannot Be Used For Modeling

In [75]:
feature_dataset = modeling_base.dropna().copy()

In [76]:
feature_dataset.shape

(41370, 24)

# Check Final Dataset

In [77]:
feature_dataset.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,...,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
19,2018-01-31,AAPL,39.138020,0.002755,1.46,0.0146,248.859,0.000000,Apple Inc.,Information Technology,...,-0.001378,0.002755,0.000008,0.006855,0.000087,0.011013,0.009462,40.695438,0.961730,0.022707
20,2018-02-01,AAPL,39.219841,0.002091,1.48,0.0148,249.529,0.002692,Apple Inc.,Information Technology,...,-0.001265,0.002091,0.000004,0.006951,0.000087,0.010065,0.009490,40.643427,0.964974,0.022726
21,2018-02-02,AAPL,37.518093,-0.043390,1.48,0.0148,249.529,0.002692,Apple Inc.,Information Technology,...,-0.003667,0.043390,0.001883,0.008888,0.000180,0.019425,0.013250,40.496978,0.926442,0.019948
22,2018-02-05,AAPL,36.580715,-0.024985,1.51,0.0151,249.529,0.002692,Apple Inc.,Information Technology,...,-0.005485,0.024985,0.000624,0.009568,0.000205,0.019936,0.013567,40.280635,0.908146,0.018715
23,2018-02-06,AAPL,38.109501,0.041792,1.52,0.0152,249.529,0.002692,Apple Inc.,Information Technology,...,-0.003210,0.041792,0.001747,0.011472,0.000292,0.032292,0.017207,40.148329,0.949218,0.017050


In [78]:
feature_dataset.isna().sum()

Date                          0
ticker                        0
adjusted_close                0
daily_return                  0
risk_free_rate_pct            0
risk_free_rate_decimal        0
cpi_index                     0
cpi_pct_change                0
company_name                  0
gics_sector                   0
asset_type                    0
return_lag_1                  0
return_lag_5                  0
rolling_return_5d             0
rolling_return_20d            0
abs_return                    0
squared_return                0
rolling_abs_return_20d        0
rolling_squared_return_20d    0
rolling_volatility_5d         0
rolling_volatility_20d        0
moving_avg_20d                0
price_to_moving_avg_20d       0
future_volatility_20d         0
dtype: int64

# Save Final Notebook

In [79]:
feature_dataset.to_csv(
    FEATURE_DATA_PATH / "feature_engineered_dataset.csv",
    index=False
)